# 00 - Diagnostico, alcance e ingesta reproducible

## Resumen amplio

Este notebook es el punto de entrada tecnico del proyecto. Su funcion no es entrenar modelos, sino dejar una base de datos canonica y verificable a partir de **LexGLUE / ECtHR Task B**. La tarea real es multietiqueta: un caso puede activar varios articulos del Convenio Europeo de Derechos Humanos, asi que no tiene sentido convertir el problema en una clasificacion multiclase simple.

El criterio de reproducibilidad del enunciado exige que otra persona pueda reconstruir el trabajo desde el codigo. Aqui la reconstruccion depende de dos piezas: `schema.sql`, que define SQLite, y los notebooks, que materializan datos, modelos, metricas y figuras. Esta libreta fuerza la reconstruccion desde el esquema para demostrar que la base no es un artefacto manual.

Tambien fija el alcance: el sistema apoya el analisis inicial de casos ante el TEDH, sugiere articulos candidatos y precedentes, pero no decide sentencias ni sustituye el razonamiento juridico.

## Indice

1. Preparacion reproducible y rutas.
2. Reconstruccion desde `schema.sql`.
3. Explicacion tabla a tabla del esquema.
4. Verificaciones de integridad.
5. Ejemplos reales del dataset.
6. Matriz multietiqueta real.
7. Evidencias guardadas.


![Esquema especifico generado con Image Gen](artifacts/figures/generated/notebook_00_ingestion_schema_v2.png)

**Lectura del esquema.** La imagen resume la ingesta reproducible: documentos juridicos reales, separacion en parrafos, normalizacion y materializacion en SQLite con comprobaciones de integridad. La imagen es conceptual; las cifras y conclusiones se calculan en las celdas del notebook con datos reales.


In [1]:
from pathlib import Path
import json, sqlite3, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
import project_utils as pu
warnings.filterwarnings('ignore')
pu.configure()
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 180)
print(f'Raiz del proyecto: {pu.ROOT}')
print(f'Base SQLite: {pu.DB}')


Raiz del proyecto: C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado
Base SQLite: C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado\data\interim\metadata.db


## 1. Reconstruccion desde `schema.sql`

La siguiente celda fuerza la reconstruccion de `metadata.db`. Si se borra la base, este notebook vuelve a crearla leyendo `schema.sql` y cargando LexGLUE desde Hugging Face o desde el cache local.


In [2]:
FORCE_REBUILD_FROM_SCHEMA = True
status = pu.materialize_database(force=FORCE_REBUILD_FROM_SCHEMA)
display(pd.DataFrame([status]).drop(columns=['split_counts']))
display(pd.DataFrame(status['split_counts']))


,n_cases,n_paragraphs,n_articles,n_positive_case_article_pairs,db_path,schema_path
0,11000,262580,10,15991,C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado\data\interim\metadata.db,C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado\schema.sql


,split,n_cases
0,test,1000
1,train,9000
2,validation,1000


## 2. Esquema relacional explicado

- `cases`: una fila por caso, texto unido y metadatos de longitud.
- `case_paragraphs`: parrafos originales para trazabilidad.
- `articles`: catalogo de las diez etiquetas de ECtHR Task B.
- `case_labels`: relacion multietiqueta caso-articulo.
- `experiment_runs`: configuracion y metricas de experimentos.
- `predictions`: scores y predicciones por caso.
- `explanations`: artefactos de XAI.

La tabla clave es `case_labels`: un caso puede aparecer varias veces, una por articulo positivo. Esta estructura corresponde a la matriz binaria `Y` que se usa en modelado.


In [3]:
with pu.connect_db() as conn:
    display(pd.read_sql_query("SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name", conn))
    for table in ['cases','case_paragraphs','articles','case_labels','experiment_runs','predictions','explanations']:
        print(f'\n--- {table} ---')
        display(pd.read_sql_query(f'PRAGMA table_info({table})', conn))


,name,type
0,articles,table
1,case_labels,table
2,case_paragraphs,table
3,cases,table
4,experiment_runs,table
5,explanations,table
6,predictions,table
7,v_case_label_codes,view
8,v_case_label_summary,view



--- cases ---


,cid,name,type,notnull,dflt_value,pk
0,0,case_id,TEXT,0,None,1
1,1,task,TEXT,1,None,0
2,2,split,TEXT,1,None,0
3,3,year,INTEGER,0,None,0
4,4,text_full,TEXT,1,None,0
5,5,n_paragraphs,INTEGER,1,None,0
6,6,n_tokens,INTEGER,1,None,0



--- case_paragraphs ---


,cid,name,type,notnull,dflt_value,pk
0,0,case_id,TEXT,1,None,1
1,1,paragraph_idx,INTEGER,1,None,2
2,2,paragraph_text,TEXT,1,None,0


...

,cid,name,type,notnull,dflt_value,pk
0,0,run_id,TEXT,0,None,1
1,1,stage,TEXT,1,None,0
2,2,model_name,TEXT,1,None,0
3,3,config_json,TEXT,1,None,0
4,4,metrics_json,TEXT,1,None,0
5,5,created_at,TEXT,1,None,0
6,6,git_commit,TEXT,0,None,0



--- predictions ---


,cid,name,type,notnull,dflt_value,pk
0,0,run_id,TEXT,1,None,1
1,1,case_id,TEXT,1,None,2
2,2,y_true_json,TEXT,1,None,0
3,3,y_pred_json,TEXT,1,None,0
4,4,scores_json,TEXT,0,None,0



--- explanations ---


,cid,name,type,notnull,dflt_value,pk
0,0,run_id,TEXT,1,None,1
1,1,case_id,TEXT,1,None,2
2,2,method,TEXT,1,None,3
3,3,artifact_path,TEXT,0,None,0
4,4,summary_json,TEXT,1,None,0


## 3. Verificaciones de integridad

Estas comprobaciones evitan entrenar sobre una base corrupta: no debe haber textos vacios, etiquetas huerfanas ni parrafos sin caso. Tambien se valida el volumen real del dataset.


In [4]:
with pu.connect_db() as conn:
    checks = pd.DataFrame([
        {'check':'casos totales','value':conn.execute('SELECT COUNT(*) FROM cases').fetchone()[0]},
        {'check':'casos sin texto','value':conn.execute("SELECT COUNT(*) FROM cases WHERE TRIM(text_full) = ''").fetchone()[0]},
        {'check':'parrafos totales','value':conn.execute('SELECT COUNT(*) FROM case_paragraphs').fetchone()[0]},
        {'check':'articulos catalogados','value':conn.execute('SELECT COUNT(*) FROM articles').fetchone()[0]},
        {'check':'pares positivos caso-articulo','value':conn.execute('SELECT COUNT(*) FROM case_labels').fetchone()[0]},
        {'check':'etiquetas huerfanas','value':conn.execute('SELECT COUNT(*) FROM case_labels cl LEFT JOIN articles a ON a.article_id=cl.article_id WHERE a.article_id IS NULL').fetchone()[0]},
        {'check':'parrafos huerfanos','value':conn.execute('SELECT COUNT(*) FROM case_paragraphs cp LEFT JOIN cases c ON c.case_id=cp.case_id WHERE c.case_id IS NULL').fetchone()[0]},
    ])
    split_counts = pd.read_sql_query('SELECT split, COUNT(*) AS n_cases FROM cases GROUP BY split ORDER BY split', conn)
    article_catalog = pd.read_sql_query('SELECT * FROM articles ORDER BY CAST(article_id AS INTEGER)', conn)
display(checks); display(split_counts); display(article_catalog)


,check,value
0,casos totales,11000
1,casos sin texto,0
2,parrafos totales,262580
3,articulos catalogados,10
4,pares positivos caso-articulo,15991
5,etiquetas huerfanas,0
6,parrafos huerfanos,0


,split,n_cases
0,test,1000
1,train,9000
2,validation,1000


,article_id,article_code,description
0,0,2,Derecho a la vida
1,1,3,Prohibicion de tortura o tratos inhumanos/degradantes
2,2,5,Derecho a la libertad y seguridad
3,3,6,Derecho a un proceso equitativo
4,4,8,Derecho al respeto de la vida privada y familiar
5,5,9,"Libertad de pensamiento, conciencia y religion"
6,6,10,Libertad de expresion
7,7,11,Libertad de reunion y asociacion
8,8,14,Prohibicion de discriminacion
9,9,P1-1,Proteccion de la propiedad


### Interpretacion de las comprobaciones

Estas comprobaciones son parte del trabajo, no una formalidad. Si hubiera etiquetas huerfanas, modelos y metricas no estarian midiendo los articulos esperados. Si hubiera parrafos huerfanos, perderiamos trazabilidad entre el texto original y `text_full`. Si hubiera casos sin texto, el vectorizador podria entrenar sobre ruido.

Tambien se ve que `year` no esta disponible en esta version del dataset. Por rigor, el proyecto no inventa una variable temporal a partir de fechas mencionadas en los hechos, porque esas fechas pueden referirse a sucesos, recursos o informes, no a la fecha de sentencia.


## 4. Ejemplos reales

Los ejemplos didacticos salen del propio dataset. `most_labels` muestra por que la salida debe ser multietiqueta; `longest` muestra el problema de texto largo; las etiquetas 9 y P1-1 muestran casos escasos o tematicamente especificos.


In [5]:
examples = pu.real_example_cases()
for name, df in examples.items():
    print(f'\n### {name}')
    display(df)



### most_labels


,case_id,split,year,n_tokens,n_labels,article_codes,excerpt
0,ecthr_task_b_train_004673,train,None,3738,7,"3, 5, 6, 8, 11, 14, P1-1","7. The applicant was born in 1939 and lives in Nicosia. 8. The applicant claimed that her former husband, Mr Ioannis Vrahimis, had been the director and shareholder of a com..."
1,ecthr_task_b_train_004661,train,None,3326,7,"3, 5, 6, 8, 11, 14, P1-1",7. The applicant was born in 1964 and lives in Larnaca. 8. The applicant claimed that his home had been in the village of Marathovounos in the District of Famagusta (norther...
2,ecthr_task_b_train_004815,train,None,2951,7,"3, 5, 6, 8, 11, 14, P1-1",7. The applicant was born in 1950 and lives in Nicosia. 8. The applicant claimed that she had had her home as well as other immovable property in the occupied part of Nicosi...
3,ecthr_task_b_train_004669,train,None,2386,7,"3, 5, 6, 8, 11, 14, P1-1","7. The applicant was born in 1933 and lives in Limassol. 8. The applicant claimed that in 1952, when she was 19 years' old, she had permanently settled in Famagusta (norther..."
4,ecthr_task_b_train_000951,train,None,7113,6,"2, 3, 5, 8, 10, 14","10. The applicants are six women from northern Iraq, born in 1950, 1970, 1951, 1939, 1949 and 1947 respectively. The first applicant brought the application on her own behalf ..."
5,ecthr_task_b_train_000826,train,None,6522,6,"2, 3, 5, 8, 9, 14",9. The applicants were born in 1933 and 1974 respectively and live in Diyarbakır. 10. The facts surrounding the arrest and subsequent death of Mehmet Şah İkincisoy (“Mehmet ...
6,ecthr_task_b_train_000920,train,None,3749,6,"3, 5, 6, 8, 14, P1-1","9. The applicant was born in 1952 and lives in Switzerland. Until December 1993 she lived in the Düzcealan village, attached to the Tatvan District in the province of Bitlis. ..."
7,ecthr_task_b_train_000761,train,None,3453,6,"3, 5, 6, 8, 14, P1-1","9. The applicant was born in 1933 and lives in Diyarbakır. Until the end of 1993, the applicant lived in the village of Akdoruk, attached to the Kulp District in the province ..."



### longest


,case_id,split,year,n_paragraphs,n_tokens,excerpt
0,ecthr_task_b_train_000588,train,None,558,35780,10. The case concerns events in November and December 1993 when the applicants were taken into custody for questioning about their alleged links with the PKK (the Kurdish Work...
1,ecthr_task_b_train_007600,train,None,370,34258,7. Mr Khodorkovskiy (the first applicant) was born in 1963. He is currently serving a prison sentence in a penal colony in the Karelia Region. Mr Lebedev (the second applicant...
2,ecthr_task_b_train_006273,train,None,301,22589,"6. The applicant, OAO Neftyanaya Kompaniya YUKOS, was a publicly-traded private open joint-stock company incorporated under the laws of Russia. It was registered in Nefteyugan..."
3,ecthr_task_b_train_008750,train,None,138,22499,"7. Although the present case primarily relates to the anti-riot operation of 26 September 1999, that operation in fact constituted the climax of a series of long-standing conf..."
4,ecthr_task_b_train_001122,train,None,201,22106,"52. The applicants, Mr Abdul-Vakhab Shamayev, Mr Rizvan (or Rezvan) Vissitov, Mr Khusein Aziev, Mr Adlan (or Aslan) Adayev (or Adiev), Mr Khusein Khadjiev, Mr Ruslan Gelogayev..."
5,ecthr_task_b_train_004636,train,None,71,21692,"7. “Teleradio-Moldova” (TRM) was created by Presidential decree as a State-owned company on 11 March 1994, out of the previously existing State broadcasting body. TRM's statut..."
6,ecthr_task_b_train_000375,train,None,235,21283,"11. The applicant was born in 1973 and lives in Derik, Turkey. 12. The case concerns the circumstances of the death of the applicant's brother, Mr Yakup Aktaş. According to ..."
7,ecthr_task_b_train_005125,train,None,91,20760,"9. The facts of the case and the relevant legal framework may be summarised as follows. 10. On 20 March 2003 a coalition of armed forces (the Multinational Force or “MNF”), ..."



### rare_article_9


,case_id,split,year,n_tokens,article_code,excerpt
0,ecthr_task_b_train_008242,train,None,9495,9,"5. The applicants are all Jehovah’s Witnesses, except for the ninth applicant, M. Kvergelidze. Their application to the Court is based on thirty cases of alleged violence and ..."
1,ecthr_task_b_train_007367,train,None,9193,9,5. The applicants were born in 1964 and 1972 respectively. The first applicant’s whereabouts are unknown. The second applicant lives in Tyumen. 6. The applicants are members...
2,ecthr_task_b_train_008094,train,None,8351,9,12. The applicant was born in 1937 and lives in Cieza. He is married and the father of five children. 13. He was ordained as a priest in 1961. In 1984 he applied to the Vati...
3,ecthr_task_b_validation_000022,validation,None,7730,9,8. The applicant is a Moldovan national belonging to the German ethnic minority. He was born in 1978 and lived in Tiraspol until 2010. Since 2011 he has been an asylum-seeker ...
4,ecthr_task_b_test_000282,test,None,6679,9,"5. The applicant was born in 1973. She grew up in Šiauliai, which in 2003, the time relevant in this case, had about 130,000 inhabitants. She currently lives in Vilnius. 6. ..."



### property_article


,case_id,split,year,n_tokens,article_code,excerpt
0,ecthr_task_b_train_007600,train,None,34258,P1-1,7. Mr Khodorkovskiy (the first applicant) was born in 1963. He is currently serving a prison sentence in a penal colony in the Karelia Region. Mr Lebedev (the second applicant...
1,ecthr_task_b_train_006273,train,None,22589,P1-1,"6. The applicant, OAO Neftyanaya Kompaniya YUKOS, was a publicly-traded private open joint-stock company incorporated under the laws of Russia. It was registered in Nefteyugan..."
2,ecthr_task_b_train_008750,train,None,22499,P1-1,"7. Although the present case primarily relates to the anti-riot operation of 26 September 1999, that operation in fact constituted the climax of a series of long-standing conf..."
3,ecthr_task_b_train_005568,train,None,17632,P1-1,"7. The applicant was born in 1950 and lives in Grozny. 8. According to the applicant, he experiences difficulties in reconstructing the events during and following his deten..."
4,ecthr_task_b_train_006075,train,None,14986,P1-1,7. The applicants are residents of the town of Urus-Martan in the Chechen Republic. 8. At the material time all the applicants lived at various addresses in Urus-Martan. 9....


## 5. Matriz multietiqueta

La matriz `Y` tiene una fila por caso y una columna por articulo. Un `1` significa que LexGLUE marca ese articulo como positivo para el caso.


In [6]:
cases, labels, articles = pu.load_cases_labels()
y = pu.multilabel_matrix(labels, cases['case_id'], articles)
display(y.head())
display(y.sum(axis=1).describe().to_frame('articulos_positivos_por_caso'))


,2,3,5,6,8,9,10,11,14,P1-1
case_id,,,,,,,,,,
ecthr_task_b_test_000000,0,0,0,0,0,0,1,0,0,0
ecthr_task_b_test_000001,0,0,0,0,1,0,0,0,0,0
ecthr_task_b_test_000002,0,0,0,1,0,0,0,0,0,0
ecthr_task_b_test_000003,0,0,0,1,0,0,0,0,0,0
ecthr_task_b_test_000004,0,1,0,1,0,0,0,0,0,0


,articulos_positivos_por_caso
count,11000.000000
mean,1.453727
std,0.757329
min,0.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,7.000000


## 6. Resultado del notebook

Quedan preparados `data/interim/metadata.db` y `artifacts/reports/ingestion_status.json`. Los demas notebooks pueden ejecutarse directamente porque llaman a la misma materializacion si la base falta.
